# Crear deployment job

En Databricks crea y conecta el job; en local genera un manifiesto del DAG.

In [ ]:
import os
from iris_mlflow_utils import build_deployment_config, build_runtime_config, detect_runtime, write_manifest

runtime_mode = detect_runtime()
config = build_runtime_config(model_slug='random_forest')
deployment_config = build_deployment_config()
if runtime_mode == 'local':
    manifest_path = config.deployment_manifest_path.with_name('local_deployment_job_manifest.json')
    manifest = {
        'runtime': 'local',
        'status': 'simulated',
        'job_name': deployment_config.job_name,
        'tasks': ['evaluate_model', 'Approval_Check', 'deploy_model'],
        'model_name': deployment_config.model_name,
        'deployment_skipped': True,
    }
    write_manifest(manifest_path, manifest)
    print(manifest)
else:
    from databricks.sdk import WorkspaceClient
    from mlflow.tracking import MlflowClient
    workspace = WorkspaceClient()
    notebook_root = deployment_config.notebook_root.rstrip('/')
    cluster_id = os.getenv('IRIS_DEPLOYMENT_CLUSTER_ID', '').strip()
    if not cluster_id:
        raise ValueError('Configura IRIS_DEPLOYMENT_CLUSTER_ID para crear el job.')
    service_principal = os.getenv('IRIS_DEPLOYMENT_SERVICE_PRINCIPAL', '').strip()
    tasks = [
        {'task_key': 'evaluate_model', 'existing_cluster_id': cluster_id, 'notebook_task': {'notebook_path': f'{notebook_root}/evaluate_model'}},
        {'task_key': 'Approval_Check', 'existing_cluster_id': cluster_id, 'depends_on': [{'task_key': 'evaluate_model'}], 'notebook_task': {'notebook_path': f'{notebook_root}/approval'}, 'max_retries': 0},
        {'task_key': 'deploy_model', 'existing_cluster_id': cluster_id, 'depends_on': [{'task_key': 'Approval_Check'}], 'notebook_task': {'notebook_path': f'{notebook_root}/deploy_model'}},
    ]
    settings = {'name': deployment_config.job_name, 'max_concurrent_runs': 1, 'tasks': tasks, 'parameters': [{'name': 'model_name', 'default': deployment_config.model_name}, {'name': 'model_version', 'default': ''}]}
    if service_principal:
        settings['run_as'] = {'service_principal_name': service_principal}
    created_job = workspace.api_client.do('POST', '/api/2.1/jobs/create', body=settings)
    job_id = created_job['job_id']
    registry = MlflowClient(registry_uri='databricks-uc')
    registry.update_registered_model(name=deployment_config.model_name, deployment_job_id=str(job_id))
    print({'runtime': runtime_mode, 'job_id': job_id, 'model_name': deployment_config.model_name, 'endpoint': deployment_config.endpoint_name})
